# EndzoneAPI: a season of football in five minutes

This notebook loads the free EndzoneAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: EndzoneAPI (endzoneapi.com), under the [EndzoneAPI Sample Data Licence](https://github.com/sports-data-hq/endzoneapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/endzoneapi-mcp/releases/download/season-pack/endzoneapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('nfl', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## The final standings

In [ ]:
t('nfl', 'standings').head(10)

## Quarterbacks by EPA per dropback, minimum 300 dropbacks

In [ ]:
q = t('nfl', 'qb-epa')
q[q.dropbacks >= 300].sort_values('epa_per_dropback', ascending=False)[['qb', 'team', 'dropbacks', 'epa_per_dropback', 'success_rate', 'cpoe']].head(10)

## Offense against defense, EPA per play

In [ ]:
e = t('nfl', 'team-epa')
ax = e.plot.scatter(x='epa_per_play', y='defense_epa_per_play_allowed', figsize=(7, 6), title='Offensive EPA/play vs defensive EPA/play allowed (lower is better)')
for _, r in e.iterrows():
    ax.annotate(r['team'], (r['epa_per_play'], r['defense_epa_per_play_allowed']), fontsize=7)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the EndzoneAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=endzoneapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/nfl/get_standings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the football skills from https://github.com/sports-data-hq/endzoneapi-mcp.